# Diabetes Prediction — Logistic Regression

**Goal:** Predict whether a patient has **diabetes** (binary: 0/1) from demographic and health features — gender, age, hypertension, heart disease, smoking history, BMI, HbA1c level, and blood glucose level — using Logistic Regression.

**Dataset:** `diabetes_prediction_dataset.csv` — 100,000 patient records, 9 columns (2 categorical, 6 numeric/binary, 1 target).

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
)

import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid")

## 2. Load & Clean Data

In [ ]:
df = pd.read_csv("diabetes_prediction_dataset.csv")

print("Shape:", df.shape)
print("\nMissing values:\n", df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())

df.head()

In [ ]:
# Remove duplicate records
df.drop_duplicates(inplace=True)
print("Shape after removing duplicates:", df.shape)

df.info()

**Insight:** No missing values, but **3,854 duplicate rows** were found and removed (100,000 → 96,146). The dataset is otherwise clean and well-typed.

In [ ]:
numeric_cols = ["age", "bmi", "HbA1c_level", "blood_glucose_level"]
categorical_cols = ["gender", "smoking_history", "heart_disease", "hypertension"]

df.describe()

## 3. Exploratory Data Analysis

In [ ]:
# Distribution of numeric features
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    sns.histplot(df[col], kde=True, bins=20, ax=axes[i])
    axes[i].set_title(f"Distribution of {col}")
    axes[i].set_xlabel(col)
    axes[i].set_ylabel("Frequency")

plt.tight_layout()
plt.show()

In [ ]:
# Count of categorical features
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for i, col in enumerate(categorical_cols):
    sns.countplot(data=df, x=col, ax=axes[i])
    axes[i].set_title(f"Count of {col}")
    axes[i].set_xlabel(col)
    axes[i].set_ylabel("Count")

plt.tight_layout()
plt.show()

In [ ]:
# Diabetes distribution grouped by gender, hypertension, heart disease
grouped_cols = ["gender", "hypertension", "heart_disease"]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for i, col in enumerate(grouped_cols):
    sns.countplot(data=df, x=col, hue="diabetes", ax=axes[i])
    axes[i].set_title(f"Diabetes Distribution by {col}")
    axes[i].set_xlabel(col)
    axes[i].set_ylabel("Count")
    axes[i].legend(title="Diabetes")

fig.delaxes(axes[3])
plt.tight_layout()
plt.show()

**Insight:** Diabetes is more common among patients with **hypertension** and **heart disease**, hinting these will be useful predictors. Gender shows only a mild difference in diabetes rates.

In [ ]:
# Boxplots to check for outliers
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    sns.boxplot(data=df, y=col, ax=axes[i])
    axes[i].set_title(f"Boxplot of {col}")
    axes[i].set_ylabel("")

plt.tight_layout()
plt.show()

In [ ]:
# Cap outliers using the IQR method
selected_cols = ["bmi", "HbA1c_level", "blood_glucose_level"]

for col in selected_cols:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound, upper_bound = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    df[col] = df[col].clip(lower=lower_bound, upper=upper_bound)

print("Shape after outlier handling:", df.shape)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    sns.boxplot(data=df, y=col, ax=axes[i])
    axes[i].set_title(f"Boxplot of {col} (after capping)")
    axes[i].set_ylabel("")

plt.tight_layout()
plt.show()

In [ ]:
# Correlation between numeric features
plt.figure(figsize=(8, 6))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()

**Insight:** Outliers in `bmi`, `HbA1c_level`, and `blood_glucose_level` were **capped (not removed)** using the IQR method, preserving all 96,146 rows. `HbA1c_level` and `blood_glucose_level` show the strongest correlation with `diabetes`.

## 4. Preprocessing

Split the data (stratified, to keep class balance), one-hot encode categorical columns, then scale numeric features — all fit on train and applied to test to avoid data leakage.

In [ ]:
categorical_cols = ["gender", "smoking_history"]


def split_data(df, target="diabetes", test_size=0.2, random_state=42):
    X = df.drop(columns=[target])
    y = df[target]
    return train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )


def encode_features(X_train, X_test, categorical_cols):
    encoder = OneHotEncoder(sparse_output=False, drop="first")
    encoder.fit(X_train[categorical_cols])

    def _encode(X):
        encoded = pd.DataFrame(
            encoder.transform(X[categorical_cols]),
            columns=encoder.get_feature_names_out(categorical_cols),
            index=X.index,
        )
        return pd.concat([X.drop(columns=categorical_cols), encoded], axis=1)

    return _encode(X_train), _encode(X_test), encoder


def scale_features(X_train, X_test):
    scaler = StandardScaler()
    scaler.fit(X_train)
    return scaler.transform(X_train), scaler.transform(X_test)


X_train, X_test, y_train, y_test = split_data(df)
X_train_encoded, X_test_encoded, encoder = encode_features(X_train, X_test, categorical_cols)
X_train_scaled, X_test_scaled = scale_features(X_train_encoded, X_test_encoded)

print("Train shape:", X_train_scaled.shape)
print("Test shape:", X_test_scaled.shape)

## 5. Train the Logistic Regression Model

In [ ]:
logistic_model = LogisticRegression(max_iter=1000)
logistic_model.fit(X_train_scaled, y_train)

y_pred = logistic_model.predict(X_test_scaled)

train_score = logistic_model.score(X_train_scaled, y_train)
test_score = logistic_model.score(X_test_scaled, y_test)

print(f"Train Score: {train_score:.4f}")
print(f"Test Score:  {test_score:.4f}")

## 6. Model Evaluation

In [ ]:
def evaluate_model(y_test, y_pred):
    metrics = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred),
    }
    for name, value in metrics.items():
        print(f"{name}: {value:.4f}")
    print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
    return metrics


evaluate_model(y_test, y_pred)

In [ ]:
# Confusion matrix heatmap
plt.figure(figsize=(6, 5))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, cmap="Blues", fmt="g")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
# Class distribution (imbalance check)
plt.figure(figsize=(6, 5))
ax = sns.countplot(x="diabetes", data=df)
ax.bar_label(ax.containers[0])
plt.xlabel("Diabetes")
plt.ylabel("Count")
plt.title("Class Distribution")
plt.show()

**Insight:** The baseline model reaches **95.8% accuracy**, but the dataset is **imbalanced** (far fewer diabetic cases). Recall on the diabetic class is only **0.63** — meaning ~37% of actual diabetic patients are missed, which matters more than raw accuracy in a medical context.

## 7. Handling Class Imbalance

In [ ]:
# Re-weight the minority (diabetic) class to reduce false negatives
logistic_model = LogisticRegression(class_weight={0: 1, 1: 1.8}, max_iter=1000)
logistic_model.fit(X_train_scaled, y_train)

y_pred = logistic_model.predict(X_test_scaled)

train_score = logistic_model.score(X_train_scaled, y_train)
test_score = logistic_model.score(X_test_scaled, y_test)

print(f"Train Score: {train_score:.4f}")
print(f"Test Score:  {test_score:.4f}")

evaluate_model(y_test, y_pred)

**Insight:** Weighting the diabetic class (`1.8x`) trades a small accuracy drop (95.8% → 95.3%) for a **better recall (0.63 → 0.69)** — catching more true diabetic cases at the cost of slightly more false positives. This is a worthwhile trade-off for a health screening use case.

## 8. Summary

- 96,146 clean records after removing duplicates; outliers capped, not dropped.
- `HbA1c_level`, `blood_glucose_level`, `hypertension`, and `heart_disease` are the strongest signals for diabetes.
- Baseline Logistic Regression: **95.8% accuracy**, but low recall (0.63) due to class imbalance.
- Class-weighted model improves recall to **0.69** with only a small accuracy trade-off — a better fit when missing a diabetic case is costlier than a false alarm.